# **분할 정복 & 동적 프로그래밍**



---



**(코랩에서)한글 폰트 지정하는 방법**

In [ ]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

# 코랩에서 위 코드를 실행시킨 후  반드시 코랩 메뉴: "런타임>세션 다시 시작" 합니다.

In [ ]:
# korean font
# Colab: NanumGothic, Mac: AppleGothic, 윈도우: Malgun Gothic
fontname = 'NanumGothic'
figsize = (5, 3)
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': fontname,        # (코랩)한글 폰트
                     'font.size': 12,
                     'figure.figsize': figsize,
                     'axes.unicode_minus':  False }) # 폰트 설정



------------------------------

# **1. 분할 정복**(Divide and Conquer, **DC**)



---



## **1-1. 분할 정복 개요**

강의자료 참고



---



## **1-2. 분할 정복 구조**

**분할 정복 단계**
- **분할(Divide)**: 원래 문제를 더 작은 부분 문제들로 나눈다.
- **정복(Conquer)**: 각 부분 문제를 보통 재귀적으로 해결하여 부분 해를 만든다.
- **결합(Combine)**: 부분 문제들의 해를 합쳐서 원래 문제의 최종 해를 구한다.

### **@Merge sort의 분할 정복 원리**

- **분할(Divide)**:  입력 리스트를 같은 크기의 2개의 부분 리스트로 분할
- **정복(Conquer)**: 부분 리스트를 정렬, 부분 리스트의 크기가 충분히 작지 않으면 순환 호출을 이용하여 다시 분할 정복 기법 적용, 리스트의 크기가 1이면 이미 정복(정렬)된 것.
- **결합(Combine)**: 정렬된 부분 리스트들을 하나의 배열에 통합

In [ ]:
def merge(left, right):
    print('[정복] : ->left:', list(left), 'right:', list(right))
    merged = []
    l_idx, r_idx = 0, 0

    # 두 부분 배열을 비교하면서 작은 값을 선택하여 병합
    while l_idx < len(left) and r_idx < len(right):
        if left[l_idx] < right[r_idx]:
            merged.append(left[l_idx])
            l_idx += 1
        else:
            merged.append(right[r_idx])
            r_idx += 1

    # 남은 요소들을 추가
    merged += left[l_idx:]
    merged += right[r_idx:]
    print('[결합] : ->merged:', merged),
    return merged


def merge_sort(arr):
    print('[분할] : ', arr)
    if len(arr) <= 1:
        return arr

    # 배열을 반으로 나눔
    mid = len(arr) // 2
    # 각 부분 배열에 대해 재귀적으로 병합 정렬 수행
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    # 정렬된 부분 배열을 병합
    return merge(left, right)


# 예시 사용
arr = [12,31,25,8,32,17,40,42]
print("✅정렬된 배열:", merge_sort(arr))




---



## **1-3. 분할 정복 예시**

### **예제 : 피보나치 수열**

- **분할(Divide)**:  fib(n)을 계산하기 위해, 문제를  두 개의 하위 문제 fib(n-1), fib(n-2) 로 분할
- **정복(Conquer)**: 각각 재귀적으로 호출하여 해 구함
- **결합(Combine)**: 각각의 결과를 합쳐서 fib(n) 해 구함.

In [ ]:
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)    # 중복 문제가 발생할 수 있다
fibonacci(10)

### **예제 : 최근접 쌍의 거리 문제**

- **분할(Divide)**:  x축 기준 중간에서 좌/우 반으로 나눔.
- **정복(Conquer)**: 각각 최근접 쌍 거리 구함 (재귀)
- **결합(Combine)**: 경계선 근처에서 좌우 교차 쌍((strip) 확인 후 더 가까운 거리 찾음.

In [ ]:
import math

# 거리 계산
def euclidean(p1, p2):
    distance = math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)
    # distance = math.hypot(p1[0] - p2[0], p1[1] - p2[1]) # hypotenuse(빗변)
    return distance


# 브루트 포스: 3개 이하일 때 사용
def brute_force(points):
    min_d = float('inf')
    for i in range(len(points)):
        for j in range(i + 1, len(points)):
            d = euclidean(points[i], points[j])
            if d < min_d:
                min_d = d
    return min_d


# strip 내 거리 계산
def strip_closest(strip, d_min):
    strip.sort(key=lambda p: p[1])  # y 좌표 기준 정렬
    min_d = d_min
    for i in range(len(strip)):
        j = i + 1
        while j < len(strip) and (strip[j][1] - strip[i][1]) < min_d:
            d = euclidean(strip[i], strip[j])
            if d < min_d:
                min_d = d
            j += 1
    return min_d

# 분할 정복 구조
def closest_pair(points_sorted_x):
    n = len(points_sorted_x)
    if n <= 3:
        return brute_force(points_sorted_x)

    mid = n // 2
    left = points_sorted_x[:mid]
    right = points_sorted_x[mid:]

    # 재귀 호출
    d_l = closest_pair(left)
    d_r = closest_pair(right)
    d = min(d_l, d_r)

    # strip 영역 생성
    mid_x = points_sorted_x[mid][0]
    strip = [p for p in points_sorted_x if abs(p[0] - mid_x) < d]

    d_cross = strip_closest(strip, d)

    return min(d, d_cross)


# 테스트 예제
points = [(2, 3), (12, 30), (40, 50), (5, 1), (12, 10),(3, 4)]

# x축 기준으로 정렬
points_sorted_x = sorted(points, key=lambda p: p[0])
print('points_sorted_x : ', points_sorted_x )

# 점사이의 최근접 거리
print(f"✅ 최근접 거리: {closest_pair(points_sorted_x)}")


- 가장 가까운 거리값과 해당하는 쌍점

In [ ]:
# 최근접 쌍과, 해당 점들 출력
import math

# 두 점 사이 거리 계산
def euclidean(p1, p2):  # euclidean 거리
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

def manhattan(p1, p2): # 맨하튼 거리 (도시 블록 거리)
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

def squared_euclidean(p1, p2): # 유클리드 제곱 거리 (루트를 제거하여 연산 속도 향상 가능)
    return (p1[0] - p2[0])**2 + (p1[1] - p2[1])**2


# 브루트포스 방식 (점이 3개 이하일 때)
def brute_force(points):
    min_d = float('inf')
    pair = (None, None)
    for i in range(len(points)):
        for j in range(i + 1, len(points)):
            d = euclidean(points[i], points[j])
            if d < min_d:
                min_d = d
                pair = (points[i], points[j])
    return min_d, pair[0], pair[1]

# strip 내에서 최근접 거리 탐색
def strip_closest(strip, d_min, best_pair):
    strip.sort(key=lambda p: p[1])  # y 기준 정렬
    min_d = d_min
    pair = best_pair

    for i in range(len(strip)):
        j = i + 1
        while j < len(strip) and (strip[j][1] - strip[i][1]) < min_d:
            d = euclidean(strip[i], strip[j])
            if d < min_d:
                min_d = d
                pair = (strip[i], strip[j])
            j += 1
    return min_d, pair[0], pair[1]

# 분할 정복 핵심 함수
def closest_pair_recursive(points_sorted_x):
    n = len(points_sorted_x)
    if n <= 3:
        return brute_force(points_sorted_x)

    mid = n // 2
    mid_x = points_sorted_x[mid][0] # 중앙점의 X값
    left = points_sorted_x[:mid]    # 왼쪽 점 리스트
    right = points_sorted_x[mid:]   # 오른쪽 점 리스트

    d_left, p1_left, p2_left = closest_pair_recursive(left)     # 왼쪽 점 리스트에서 거리가 가장 가까운 거리값과 쌍점
    d_right, p1_right, p2_right = closest_pair_recursive(right) # 오른쪽 점 리스트에서 거리가 가장 가까운 거리값과 쌍점

    # 가장 가까운 점 : min(dl, dr)
    if d_left < d_right:
        d_min = d_left
        best_pair = (p1_left, p2_left)
    else:
        d_min = d_right
        best_pair = (p1_right, p2_right)

    strip = [p for p in points_sorted_x if abs(p[0] - mid_x) < d_min]
    d_cross, point1, point2 = strip_closest(strip, d_min, best_pair)

    return d_cross, point1, point2


# 테스트 예제
points = [(2, 3), (12, 30), (40, 50), (5, 1), (12, 10),(3, 4)]

# x축 기준으로 정렬
points_sorted_x = sorted(points, key=lambda p: p[0])
print('points_sorted_x : ', points_sorted_x )

# 점사이의 최근접 거리와 최근접 쌍 출력
min_distance, point1, point2 = closest_pair_recursive(points_sorted_x)
print(f"✅ 최근접 거리: {min_distance:.6f}")
print(f"✅ 최근접 쌍: {point1} 와 {point2}")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import random
import math

# ✅ 1. 거리 계산 함수
def euclidean(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

def closest_pair_brute_force(pts):
    min_dist = float('inf')
    pair = (None, None)
    for i in range(len(pts)):
        for j in range(i+1, len(pts)):
            d = euclidean(pts[i], pts[j])
            if d < min_dist:
                min_dist = d
                pair = (pts[i], pts[j])
    return min_dist, pair

# ✅ 2. 무작위 점 생성
n = 20
random.seed(42)
points = [(random.randint(0, 100), random.randint(0, 100)) for _ in range(n)]

# ✅ 3. 분할 기준 리스트
midpoints = [50 - i * 5 for i in range(6)]  # 중앙 축이 점점 왼쪽으로 이동

# ✅ 4. 애니메이션 함수 설정
fig, ax = plt.subplots(figsize=(6, 6))

def update(frame):
    ax.clear()
    midpoint = midpoints[frame]

    # 좌우 영역 분할
    left = [p for p in points if p[0] <= midpoint]
    right = [p for p in points if p[0] > midpoint]

    # 거리 계산
    dl, pl = closest_pair_brute_force(left) if len(left) >= 2 else (None, (None, None))
    dr, pr = closest_pair_brute_force(right) if len(right) >= 2 else (None, (None, None))

    # 전체 점 그리기
    ax.scatter(*zip(*points), color='black')
    ax.axvline(midpoint, color='red', linestyle='--', linewidth=2)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.set_title(f"Step {frame + 1}: Midpoint = {midpoint}", fontsize=14)

    # 왼쪽 최근접 쌍 시각화
    if dl is not None:
        x_vals = [pl[0][0], pl[1][0]]
        y_vals = [pl[0][1], pl[1][1]]
        ax.plot(x_vals, y_vals, 'b--', label='left pair')
        ax.text((x_vals[0]+x_vals[1])/2, (y_vals[0]+y_vals[1])/2 + 2,
                f"{dl:.2f}", color='blue')

    # 오른쪽 최근접 쌍 시각화
    if dr is not None:
        x_vals = [pr[0][0], pr[1][0]]
        y_vals = [pr[0][1], pr[1][1]]
        ax.plot(x_vals, y_vals, 'g--', label='right pair')
        ax.text((x_vals[0]+x_vals[1])/2, (y_vals[0]+y_vals[1])/2 + 2,
                f"{dr:.2f}", color='green')

    ax.legend(loc='upper right')

ani = animation.FuncAnimation(fig, update, frames=len(midpoints), interval=1500, repeat=False)

HTML(ani.to_jshtml())




---



# **2. 동적 프로그래밍**(Dynamic Programming, **DP**)

## 2-1. DP 개요

- 1950년대에 Richard Bellman이라는 수학자에 의해 처음 사용됨 "동적 프로그래밍(Programming)“ 또는 "동적 계획법(Planning)“이라고 부른다.
- 복잡한 문제를 하위 문제로 나누어 각각의 하위 문제들을 해결한 다음, 그 결과를 저장하고 재사용함으로써 전체 문제의 효율적인 해결을 도모하는 설계 방식을 말함
- 분할 정복(Divide and Conquer)과 유사


### 1) DP 핵심 전략

- **최적 부분 구조 (Optimal Substructure)**
    - 문제의 정답이 하위 문제의 정답을 조합하여 구할 수 있는 경우
    - 예: F(n) = F(n-1) + F(n-2)  
- **중복되는 하위 문제 (Overlapping Subproblems)**
    - 재귀적으로 푸는 경우 동일한 하위 문제를 여러 번 계산하게 됨
    - 각 하위 문제의 해결을 한 번만 수행하고 그 결과를 저장하여 재사용
    - 중복 해결 -> 메모이제이션 or 테이블 저장으로 해결

### 2) 피보나치 수열의 문제 해결 전략

- 1.분할 정복 (중복 계산의 문제)

In [ ]:
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)    # 중복 문제가 발생할 수 있다
fibonacci(10)


- 2.DP Top-down 방식, 메모이제이션(memoization) 방법 (중복 해결)

In [ ]:
def fibonacci(n):
    # -메모이제이션-
    mem = [None] * (n+1)
    if mem[n] == None:  # 처음 푸는 문제이면,
        if n <= 1:      # 풀어서 메모리에 저장함
            mem[n] = n
        else:
            mem[n] =  fibonacci(n-1) + fibonacci(n-2)
    return mem[n]
fibonacci(10)


- 3.DP Bottom-up, 테이블채우기(Tabulation) 방법 (중복 해결)
    - 선형시간에 끝나는 방법

In [ ]:
def fibonacci(n):
    # -메모이제이션-
    fib = [0] * (n+1)
    fib[1] = 1

    # -점화식-
    for i in range(2, n+1):
        fib[i] = fib[i-1] + fib[i-2]
    return fib[n]
fibonacci(10)


In [ ]:
def fibonacci(n):
    fib = [0, 1]  # 0번째와 1번째 피보나치 수를 미리 계산
    for i in range(2, n+1):
        fib.append(fib[i-1] + fib[i-2])
    return fib[n]
fibonacci(10)


## 2-2. DP 원리와 구조

### @ Rod-cutting Problem
- 목표 : 길이가 n인 쇠막대를 잘라서(자르지 않거나) 얻을 수 있는 최대의 이익 구하라
- 해결방법
    - 분할 정복(재귀)
    - DP : Top-down Memoization
    - DP : Bottom-up Tabulation

- 분할 정복(재귀)

In [ ]:
# 분할 정복
def rod_cutting_recursive(prices, n):
    if n == 0:
        return 0
    max_val = float('-inf')
    for i in range(1, n + 1):
        max_val = max(max_val, prices[i - 1] + rod_cutting_recursive(prices, n - i))
    return max_val

price = [1, 5, 8, 9, 10, 17, 17, 20]
n = len(price)
print(f"✅ 최대 이익 : {rod_cutting_recursive(price, n)}")


In [ ]:
# 분할 정복 : 절단 경로 추적
def rod_cutting_recursive_trace(prices, n):
    if n == 0:
        return 0, []
    max_val = float('-inf')
    best_cut = []
    for i in range(1, n + 1):
        val, cuts = rod_cutting_recursive_trace(prices, n - i)
        if prices[i - 1] + val > max_val:
            max_val = prices[i - 1] + val
            best_cut = [i] + cuts
    return max_val, best_cut


maxRevenue, cutList = rod_cutting_recursive_trace(price, n)
print("✅ 최대 이익:", maxRevenue)
print("✅ 자르기 방법:", cutList)

- DP : Top-down Memoization

In [ ]:
# DP : Top-down Memoization
def rod_cutting_memo(prices, n, memo=None):
    if memo is None:
        memo = [-1] * (n + 1)
    if n == 0:
        return 0
    if memo[n] >= 0:
        return memo[n]

    max_val = float('-inf')
    for i in range(1, n + 1):
        max_val = max(max_val, prices[i - 1] + rod_cutting_memo(prices, n - i, memo))
    memo[n] = max_val
    return max_val

print(f"✅ 최대 이익 : {rod_cutting_memo(price, n)}")


In [ ]:
# DP : Top-down Memoization : 절단 경로 추적
def rod_cutting_memo_trace(prices, n):
    memo = [-1] * (n + 1)
    solution = [[] for _ in range(n + 1)]

    def helper(k):
        if k == 0:
            return 0
        if memo[k] >= 0:
            return memo[k]
        max_val = float('-inf')
        for i in range(1, k + 1):
            val = prices[i - 1] + helper(k - i)
            if val > max_val:
                max_val = val
                solution[k] = [i] + solution[k - i]
        memo[k] = max_val
        return max_val

    max_profit = helper(n)
    return max_profit, solution[n]

maxRevenue, cutList = rod_cutting_memo_trace(price, n)
print("✅ 최대 이익:", maxRevenue)
print("✅ 자르기 방법:", cutList)


- DP : Bottom-up Tabulation

In [ ]:
# DP : Bottom-up Tabulation
def rod_cutting_bottom_up(prices, n):
    dp = [0] * (n + 1)
    for i in range(1, n + 1):
        max_val = float('-inf')
        for j in range(1, i + 1):
            max_val = max(max_val, prices[j - 1] + dp[i - j])
        dp[i] = max_val
    return dp[n]

print(f"✅ 최대 이익 : {rod_cutting_bottom_up(price, n)}")

In [ ]:
# DP : Bottom-up Tabulation : 절단 경로 추적
def rod_cutting_bottom_up_trace(prices, n):
    dp = [0] * (n + 1)
    cuts = [0] * (n + 1)

    for i in range(1, n + 1):
        max_val = float('-inf')
        for j in range(1, i + 1):
            if prices[j - 1] + dp[i - j] > max_val:
                max_val = prices[j - 1] + dp[i - j]
                cuts[i] = j
        dp[i] = max_val

    # 절단 경로 추적
    result = []
    while n > 0:
        result.append(cuts[n])
        n -= cuts[n]
    return dp[len(cuts) - 1], result


maxRevenue, cutList = rod_cutting_bottom_up_trace(price, n)
print("✅ 최대 이익:", maxRevenue)
print("✅ 자르기 방법:", cutList)

### **실습문제 : 철근의 최대 이익과 절단 길이 구하기**
앞에서 예로 든 철근 가격표에서 길이가 8일 때 최대 수익과 절단 길이는?

In [ ]:
prices = [1, 5, 8, 9, 10, 17, 17, 20, 24, 30]
n = 9

print("분할정복:", rod_cutting_recursive_trace(prices, n))           # 22
print("탑다운 DP:", rod_cutting_memo_trace(prices, n))               # 22
print("바텀업 DP:", rod_cutting_bottom_up_trace(prices, n))          # 22


--------------

## 2-3. DP 예시

### **예제 : Bellman-Ford Algorithm**(최단 경로 찾기)


* **핵심 아이디어** : 그래프의 모든 간선을 반복적으로 완화(relax)하여, 특정 시작 정점에서 다른 모든 정점까지의 최단 경로를 찾는 것. 음수 가중치가 있는 그래프에서도 작동하며, 음수 사이클이 존재하는 경우 이를 탐지할 수 있다.
* 연산 과정: 간선 완화는 간선의 가중치를 사용하여 정점 간의 최단 거리를 갱신하는 과정
(모든 간선을 순회하며, 현재 거리보다 더 짧은 경로가 있다면 distances 배열을 업데이트)
* 시간 복잡도: 시간 복잡도 O(V𝐸)
* 중요 특징 요소
    - **음의 가중치 허용** : 모든 간선의 가중치가 0 이상인 그래프에서 작동.
    - **사이클 감지** : 주어진 시작 정점에서 다른 모든 정점까지의 최단 경로를 찾는다.

1. (거리)초기화 **(메모이제이션)**
    - ① 모든 노드의 거리를 무한대로 설정하고,
    - ② 시작 노드의 거리는 0으로 설정
3. 간선 완화 **(점화식)**
    - ① 전체 간선에 대해서 시행한다.
    - ② 현재 노드 u의 거리에 v까지의 간선 가중치를 더한 값이 v의 현재 거리보다 작으면 v의 거리를 u의 거리에 간선 가중치를 더한 값으로 갱신
5. 음수 사이클 탐지
    - ① 모든 간선에 대해서 한 번 더 반복하여 거리가 갱신되는 노드가 있다면
    - (현재 노드 u의 거리에 v까지의 간선 가중치를 더한 값이 v의 현재 거리보다 작은 것이 존재하면) 음의 사이클이 존재한다고 판단한다.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt


# 최단 경로&거리 출력
def print_paths(shortest_path_tree, start, distances):
    def build_path(node):
        path = []
        while node is not None:
            path.append(node)
            node = shortest_path_tree[node]
        return list(reversed(path))

    print(f"✅ Shortest Path from Start({start}):")
    for node in shortest_path_tree:
        if node != start:
            path = build_path(node)
            print(f"{start} → {node} 경로: {' → '.join(path)} ({distances[node]}) ")


def draw_graph(graph, shortest_path_tree, start, pos=False):
    G = nx.DiGraph()
    for node in graph:
        for neighbor, weight in graph[node].items():
            if weight != float('inf'):
                G.add_edge(node, neighbor, weight=weight)

    if not pos:
        pos = nx.spring_layout(G)
    labels = nx.get_edge_attributes(G, 'weight')

    plt.figure(figsize=(6, 4))
    nx.draw(G, pos, with_labels=True, node_color='skyblue', node_size=1000,
            font_size=12, font_weight='bold', arrows=True)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_size=12, label_pos=0.7)   # 간선 위쪽(0.5가 중앙, 0.7은 위쪽))

    # Highlight the shortest path tree
    if start in shortest_path_tree.values():
        path_edges = [(shortest_path_tree[node], node) for node in shortest_path_tree if shortest_path_tree[node]]
        # nx.draw_networkx_edges(G, pos, edgelist=path_edges, edge_color='r', width=2.5)
        # 화살표 표시하고 싶은 경우: arrows=True, arrowstyle='-|>',arrowsize=20
        nx.draw_networkx_edges(G, pos, edgelist=path_edges, edge_color='r', width=2.5,
                               arrows=True, arrowstyle='-|>',arrowsize=20) # 화살표
                            #   connectionstyle='arc3,rad=0.3')   # 곡선

    plt.title("Graph Visualization with Shortest Path Tree")
    plt.show()


In [ ]:
def bellman_ford(graph, start):
    # 1. 거리 초기화 (메모이제이션 적용을 위한 초기화)
    distances = {node: float('inf') for node in graph}
    distances[start] = 0
    shortest_path_tree = {node: None for node in graph}

    # 2. 간선 완화
    for i in range(len(graph) - 1):
        for u, neighbors in graph.items():
            for v, weight in neighbors.items():
                new_distance = distances[u] + weight
                if new_distance < distances[v]:
                    distances[v] = new_distance
                    shortest_path_tree[v] = u

    # 3.음수 사이클 탐지
    for u, neighbors in graph.items():
        for v, weight in neighbors.items():
            if distances[u] + weight < distances[v]:
                return None, None  # Negative cycle detected

    return distances, shortest_path_tree


# 그래프 생성(딕셔너리 데이터)
def create_graph_from_edges(edges):
    graph = {}
    for u, v, w in edges:
        if u not in graph:
            graph[u] = {}
        graph[u][v] = w
        # 단방향 그래프이므로 역방향 추가 X
        if v not in graph:
            graph[v] = {}  # 노드가 연결되지 않은 경우에도 빈 딕셔너리로 초기화
    return graph


# 1. 엣지 리스트 정의
edges = [
    ('A', 'B', 11), ('A', 'C', 8), ('A', 'D', 9),
    ('B', 'E', 8), ('B', 'G', 8),
    ('C', 'F', 10),
    ('D', 'B', 3), ('D', 'C', -15),('D', 'F', 1),
    ('E', 'G', -7),
    ('F', 'H', 2),
    ('G', 'D', 12), ('G', 'H', 5),
    ('H', 'E', 4)
]


# 2. 그래프 생성 함수 호출
graph = create_graph_from_edges(edges)
print("✅ Input : ", graph)


# 3. 벨만포드 실행 및 시각화 (시작 정점에서 각 정점까지의 최단 거리 계산)
distances, shortest_path_tree = bellman_ford(graph, START)
if distances is None:
    print("✅ Negative cycle detected")
else:
    print("✅ Shortest Path Tree:", shortest_path_tree) # 최단 경로 트리 출력
    print_paths(shortest_path_tree, START, distances)# 최단 경로 트리를 이용하여 출발점 --> 노드까지의 경로 출력


# 그래프 시각화
pos = {
    'A': (-1,1),
    'B': (-1,-1),
    'C': (0,2),
    'D': (0,0),
    'E': (0,-2),
    'F': (1,1),
    'G': (1,-1),
    'H': (2,0)
}
if shortest_path_tree:
    draw_graph(graph, shortest_path_tree, START, pos)

----

### **예제 : 최장 공통 부분순서**(LCS Longest Common Subsequence)


* 두 문자열에 공통적으로 들어있는 공통 부분순서 중 가장 긴 것을 찾는 것
* 부분순서의 예
    - `<bcdb>`는 문자열 `<abcbdab>`의 부분순서
* 공통 부분순서의 예
    - `<bca>`는 문자열 `<abcbdab>`와 `<bdcaba>`의 공통 부분순서
* 최장 공통 부분순서(Longest Common Subsequence LCS)
    - 공통 부분순서들 중 가장 긴 것
    - 예: `<bcba>` 는 문자열 `<abcbdab>`와 `<bdcaba>`의 최장 공통 부분순서

#### 1) LCS의 길이(분할 정복)

In [ ]:
def lcs_recur(X, Y, m, n):
    if m == 0 or n == 0: 		# base case
        return 0
    elif X[m-1] == Y[n-1]: 		# case 1: x_m == y_n
                                # 마지막 문자가 같으면 이를 제외하고 계산한 다음 1을 더한 값이 OK
        return 1 + lcs_recur(X, Y, m-1, n-1)
    else: 						# case 2
                                # 마지막 문자가 서로 다르면, 두 가지를 계산해보고 더 큰쪽이 OK
        return max(lcs_recur(X, Y, m, n-1), lcs_recur(X, Y, m-1, n))


# LCS 추적
def lcs_recur_trace(X, Y, m, n):
    # base case: 하나라도 끝났으면 공집합
    if m == 0 or n == 0:
        return ""

    # 문자가 같으면 → 그 문자를 포함하고 왼쪽 위로 이동
    if X[m-1] == Y[n-1]:
        return lcs_recur_trace(X, Y, m-1, n-1) + X[m-1]

    # 문자가 다르면 → 두 가지 방향 중 LCS 길이가 큰 쪽을 선택
    if lcs_recur(X, Y, m, n-1) > lcs_recur(X, Y, m-1, n):
        return lcs_recur_trace(X, Y, m, n-1)
    else:
        return lcs_recur_trace(X, Y, m-1, n)


# LCS 테스트 프로그램
X = "GAME OVER"
Y = "HELLO WORLD"
print("X = ", X)
print("Y = ", Y)
print("LCS(분할 정복) 길이: ", lcs_recur(X , Y, len(X), len(Y)))
print("LCS(분할 정복) 추적: ", lcs_recur_trace(X, Y, len(X), len(Y)))


#### 2) LCS의 길이(DP)

In [ ]:
# LCS의 길이(동적 계획법)
def lcs_dp(X , Y):
    # -메모이제이션-
    m = len(X)
    n = len(Y)
    dp = [[None]*(n+1) for _ in range(m+1)] 	# 테이블 생성

    # -점화식-
    for i in range(m+1):                    # 부분 문제 해결
        for j in range(n+1):
            if i == 0 or j == 0 :		    # base case: 하나의 길이라도 0이면
                dp[i][j] = 0			        # LCS --> 0
            elif X[i-1] == Y[j-1]:	        # case1: 글자가 같으면(↖)
                dp[i][j] = dp[i-1][j-1]+1
            else:					        # case2: 글자가 다르면(↑ or ←)
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    for i in range(m+1):
        print(dp[i])

    return dp, dp[m][n]

# LCS 테이블에서 LCS 추적
def lcs_dp_trace(X, Y, dp):
    lcs = ""													# ① 최장 길이
    i = len(X)													# ② 글자 길이
    j = len(Y)													# ② 글자 길이

    while i > 0 and j > 0:
        L = dp[i][j]
        if L > dp[i][j-1] and L > dp[i-1][j]  and L > dp[i-1][j-1]:# ③ 현재 길이가 X, Y의 이전 길이가 더 길면
            i -= 1
            j -= 1
            lcs = X[i] + lcs

        elif L == dp[i][j-1] and L > dp[i-1][j]: j -= 1
        else : i -= 1

    return lcs


# LCS 테스트 프로그램
X = "GAME OVER"
Y = "HELLO WORLD"
print("X = ", X)
print("Y = ", Y)
dp, lcs = lcs_dp(X, Y)
print("LCS(동적 프로그래밍) 길이 : ", lcs)
print("LCS(동적 프로그래밍) 추적 : ", lcs_dp_trace(X, Y, dp))


In [ ]:
X = "ABCDGH"
Y = "AEDFHR"
dp, lcs = lcs_dp(X, Y)
print("LCS(동적 프로그래밍) 길이 : ", lcs)
print("LCS(동적 프로그래밍) 추적 : ", lcs_dp_trace(X, Y, dp))

In [ ]:
X = "AGGTAB"
Y = "GXTXAYB"
dp, lcs = lcs_dp(X, Y)
print("LCS(동적 프로그래밍) 길이 : ", lcs)
print("LCS(동적 프로그래밍) 추적 : ", lcs_dp_trace(X, Y, dp))

- [참고] 1차원 DP 테이블로 공간 절약

In [ ]:
def lcs_dp_1d(A, B):
    m, n = len(A), len(B)
    prev = [0] * (n + 1)

    for i in range(1, m + 1):
        curr = [0] * (n + 1)
        for j in range(1, n + 1):
            if A[i - 1] == B[j - 1]:
                curr[j] = prev[j - 1] + 1
            else:
                curr[j] = max(prev[j], curr[j - 1])
        prev = curr

    return prev[n]

X = "AGGTAB"
Y = "GXTXAYB"
lcs = lcs_dp_1d(X, Y)
print("LCS(동적 프로그래밍) 1차원 배열 사용 : ", lcs)

#### 3) [참고] 문자열 유사도 측정 방법 예

- **1.편집 거리(Levenshtein Distance)**
    - 한 문자열을 다른 문자열로 바꾸기 위해 필요한 최소한의 편집 연산(삽입, 삭제, 대체) 횟수를 구하는 방법

In [ ]:
def levenshtein_distance(X, Y):
    m, n = len(X), len(Y)
    # DP 테이블 초기화
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    # base case: 공집합과의 편집 거리
    for i in range(m + 1):
        dp[i][0] = i  # X → ''는 모두 삭제
    for j in range(n + 1):
        dp[0][j] = j  # '' → Y는 모두 삽입

    # 점화식 적용
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if X[i - 1] == Y[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]  # 문자가 같으면 편집 필요 없음
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],    # 삭제
                    dp[i][j - 1],    # 삽입
                    dp[i - 1][j - 1] # 교체
                )
    return dp[m][n]


X = "GAME OVER"
Y = "HELLO WORLD"
print("X = ", X)
print("Y = ", Y)
distance = levenshtein_distance(X, Y)
print(f"✅ 편집 거리(Levenshtein Distance) = {distance}")


- **2.자카드 유사도(Jaccard Similarity)**
    - 두 문자열을 각각 문자들의 집합 (또는 n-gram 집합)으로 보고, 두 집합 간의 유사도를 측정 방법
    - 두 집합의 교집합 크기를 합집합 크기로 나눈 값
    - (0~1 사이의 값) 1에 가까울수록 유사도가 높다.
$Jaccard(A,B)= \frac{A∪B}{A∩B}$

In [ ]:
# 문자 단위 자카드 유사도 (Char Set 기반)
def jaccard_char_similarity(str1, str2):
    set1 = set(str1)
    set2 = set(str2)

    intersection = set1.intersection(set2)
    union = set1.union(set2)

    if not union:
        return 1.0  # 둘 다 공집합이면 유사도 1로 간주

    return len(intersection) / len(union)


# n-gram 기반 자카드 유사도 (2-gram)
def jaccard_ngram_similarity(str1, str2, n=2):
    def get_ngrams(s, n):
        return {s[i:i+n] for i in range(len(s) - n + 1)}

    set1 = get_ngrams(str1, n)
    set2 = get_ngrams(str2, n)

    intersection = set1.intersection(set2)
    union = set1.union(set2)

    if not union:
        return 1.0

    return len(intersection) / len(union)

X = "GAME OVER"
Y = "HELLO WORLD"
print("X = ", X)
print("Y = ", Y)
print("✅  문자 기반 자카드 유사도:", jaccard_char_similarity(X, Y))
print("✅  2-gram 기반 자카드 유사도:", jaccard_ngram_similarity(X, Y, n=2))

- **3.코사인 유사도(Cosine Similarity)**
    - 문자열을 벡터 공간 모델(VSM)로 표현한 후, 두 벡터 간의 코사인 각도를 이용하여 유사도를 측정
    - 주로 TF-IDF(Term Frequency-Inverse Document Frequency) 등으로 문자열을 벡터화한 후 사용 (Count Vectorizer)
    - (-1~1 사이의 값) 에 가까울수록 유사도가 높다

cosine similarity= $ \frac{\overrightarrow{A}⋅\overrightarrow{B}}{\|{\overrightarrow{A}}\| \|{\overrightarrow{B}}\|}$


In [ ]:
from collections import Counter
import math

def cosine_similarity(str1, str2):
    # 소문자 처리 및 공백 제거 선택적 적용 가능
    str1 = str1.lower()
    str2 = str2.lower()

    # 문자 단위로 벡터화
    vec1 = Counter(str1)
    vec2 = Counter(str2)

    # 모든 문자 집합 (공통 키로 벡터 만들기)
    all_chars = set(vec1.keys()).union(set(vec2.keys()))

    # 두 벡터 만들기
    v1 = [vec1.get(char, 0) for char in all_chars]
    v2 = [vec2.get(char, 0) for char in all_chars]

    # 내적 계산
    dot_product = sum(a * b for a, b in zip(v1, v2))

    # 벡터 크기 계산
    norm1 = math.sqrt(sum(a * a for a in v1))
    norm2 = math.sqrt(sum(b * b for b in v2))

    if norm1 == 0 or norm2 == 0:
        return 0.0  # 둘 중 하나라도 크기가 0이면 유사도 0

    return dot_product / (norm1 * norm2)

# 예제
X = "GAME OVER"
Y = "HELLO WORLD"
print("X = ", X)
print("Y = ", Y)
similarity = cosine_similarity(X, Y)
print(f"✅ Cosine Similarity: {similarity:.4f}")


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

X = "GAME OVER"
Y = "HELLO WORLD"

vectorizer = CountVectorizer(analyzer='char')  # 문자 단위 벡터화
vectors = vectorizer.fit_transform([X, Y])

cos_sim = cosine_similarity(vectors[0], vectors[1])
print(f"✅ Cosine Similarity: {cos_sim[0][0]:.4f}")


-------------------

### **예제 : 배낭 채우기**(Knapsack problem)
- 한정된 용량의 배낭에 최대 가치를 가진 물건들을 담는 방법을 찾는 것

#### 1) 0/1 배낭 문제(분할 정복)

In [ ]:
def knapsack_recursive(weights, values, w, i):
    # 종료 조건: 물건을 다 보거나, 배낭이 가득 찼을 때
    if i < 0 or w == 0:
        return 0

    # 현재 물건을 담을 수 없으면, 이전 물건으로 넘어감
    if weights[i] > w:
        return knapsack_recursive(weights, values, w, i - 1)

    # 선택하지 않는 경우 vs 선택하는 경우 중 큰 값
    without_item = knapsack_recursive(weights, values, w, i - 1)
    with_item = values[i] + knapsack_recursive(weights, values, w - weights[i], i - 1)

    return max(without_item, with_item)

# 예시 입력
weights = [2, 5, 8, 4, 7, 6]            # 물건의 무게(wgt)
values = [60, 100, 190, 120, 200, 150]  # 물건의 가치(val)
W = 18     # 배낭의 무게(목표)

# 실행
n = len(weights)
max_value = knapsack_recursive(weights, values, W, n - 1)

# 결과 출력
print(f'목표: {W}, 무게: {weights}, 가치: {values}')
print("# 0/1 배낭문제(분할 정복)")
print("✅ 최대 가치:", max_value)


#### 2) 0/1 배낭 문제(동적 프로그래밍)

- DP, Top-down, Memoizatio

In [ ]:
def knapsack_memo(weights, values, w, i, memo):
    if i < 0 or w == 0:
        return 0
    if (i, w) in memo:
        return memo[(i, w)]

    if weights[i] > w:
        result = knapsack_memo(weights, values, w, i - 1, memo)
    else:
        result = max(
            knapsack_memo(weights, values, w, i - 1, memo),
            values[i] + knapsack_memo(weights, values, w - weights[i], i - 1, memo)
        )
    memo[(i, w)] = result
    return result

# 예시 입력
weights = [2, 5, 8, 4, 7, 6]
values = [60, 100, 190, 120, 200, 150]
W = 18

# 결과 출력
memo = {}
max_value = knapsack_memo(weights, values, W, len(weights) - 1, memo)

print(f'목표: {W}, 무게: {weights}, 가치: {values}')
print("# 0/1 배낭문제(DP,Top-down,Memoization ))")
print("✅ 최대 가치:", max_value)


- DP, Bottom-up, Tabulation

In [ ]:
# 최대 가치
def knapsack_01(weights, values, W):
    # -메모이제이션-
    n = len(weights)
    dp = [[0]*(W+1) for _ in range(n+1)]

    # -점화식-
    for i in range(1, n+1):
        for w in range(W+1):
            if weights[i-1] <= w:
                dp[i][w] = max(dp[i-1][w], dp[i-1][w - weights[i-1]] + values[i-1])
            else:
                dp[i][w] = dp[i-1][w]

    return dp[n][W]

# 예시 입력
weights = [2, 5, 8, 4, 7, 6]
values = [60, 100, 190, 120, 200, 150]
W = 18

# 결과 출력
max_value = knapsack_01(weights, values, W)
print(f'목표: {W}, 무게: {weights}, 가치: {values}')
print("# 0/1 배낭문제(DP,Bottom-up,Tabulation)")
print("✅ 최대 가치:", max_value)

In [ ]:
# 최대 가치 + (선택된 물건, 선택된 가치)
def knapsack_01(weights, values, W):
    # -메모이제이션-
    n = len(weights)
    dp = [[0]*(W+1) for _ in range(n+1)]

    # -점화식-
    for i in range(1, n+1):
        for w in range(W+1):
            if weights[i-1] <= w:
                dp[i][w] = max(dp[i-1][w], dp[i-1][w - weights[i-1]] + values[i-1])
            else:
                dp[i][w] = dp[i-1][w]

    # -역추적: 어떤 아이템을 선택했는가?-
    selected_weights = []
    selected_values = []

    i, w = n, W
    while i > 0 and w >= 0:
        if dp[i][w] != dp[i-1][w]:  # 이 물건을 선택했을 경우
            selected_weights.append(weights[i-1])
            selected_values.append(values[i-1])
            w -= weights[i-1]
        i -= 1

    selected_weights.reverse()
    selected_values.reverse()

    return dp[n][W], selected_weights, selected_values


# 예시 입력
weights = [2, 5, 8, 4, 7, 6]
values = [60, 100, 190, 120, 200, 150]
W = 18

# 실행
max_value, used_weights, used_values = knapsack_01(weights, values, W)

# 결과 출력
print(f'목표: {W}, 무게: {weights}, 가치: {values}')
print("# 0/1 배낭문제")
print("✅ 최대 가치:", max_value)
print("✅ 선택된 물건의 무게:", used_weights)
print("✅ 선택된 물건의 가치:", used_values)


-----------------------

### [참고] Fractional Knapsack (분할 가능 배낭)
배낭에 담을 수 있는 총 무게를 초과하지 않으면서 가치의 총합을 최대화하는 선택을 하라

- 탐욕 알고리즘 접근 방식
    - 1.각 아이템에 대해 단위 무게당 가치 계산
    - 2.단위 가치가 높은 아이템부터 정렬
    - 3.무게가 허용되는 한도 내에서 최대한 많이 담기
    - 4.남은 공간이 부족하면 일부만 분할해서 담기

In [ ]:
def fractional_knapsack(items, capacity):
    # items = [(value, weight), ...]
    # 단위 무게당 가치 기준으로 정렬 (내림차순)
    items.sort(key=lambda x: x[0] / x[1], reverse=True)

    total_value = 0.0
    total_weight = 0.0
    selected_items = []

    for value, weight in items:
        if capacity == 0:
            break
        if weight <= capacity:
            # 전부 담기
            selected_items.append((value, weight, 1.0))  # 1.0 = 100%
            total_value += value
            total_weight += weight
            capacity -= weight
        else:
            # 일부만 담기 (분할)
            fraction = capacity / weight
            selected_items.append((value, weight, fraction))
            total_value += value * fraction
            total_weight += capacity
            capacity = 0
            break

    # 출력
    print("✅ 선택된 아이템 (가치, 무게, 선택 비율):")
    for v, w, f in selected_items:
        print(f" - 가치: {v}, 무게: {w}, 선택비율: {f:.2f} ({v*f:.2f} 가치만큼 선택)")

    print(f"\n✅ 총 선택 무게: {total_weight:.2f}")
    print(f"✅ 총 얻은 가치: {total_value:.2f}")

    return total_value


# 예제 데이터: (value, weight)
items = [(60,2), (100,5), (190,8), (120,4), (200,7), (150,6)]
capacity = 50

fractional_knapsack(items, capacity)
